## ***A Data-Driven Analysis of Tourist Satisfaction in SriLankan Destinations Using Online Review Analytics***

# Import Libraries

In [29]:
# ==
# Core Libraries
# ==
import pandas as pd
import numpy as np

# ==
# Visualization
# ==
import matplotlib.pyplot as plt
import seaborn as sns

# ==
# Text Processing
# ==
import random
import re
import string

# ==
# Natural Language Processing
# ==
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# ==
# Feature Extraction
# ==
from sklearn.feature_extraction.text import CountVectorizer

# ==
# Sentiment Analysis
# ==
from nltk.sentiment import SentimentIntensityAnalyzer

# ==
# Topic Modeling
# ==
from gensim import corpora
from gensim.models import LdaModel

# ==
# Word Cloud
# ==
from wordcloud import WordCloud

# ==
# Display Settings
# ==
pd.set_option('display.max_columns', None)
sns.set_style("whitegrid")

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('vader_lexicon')
nltk.download('punkt_tab') # Added to resolve the LookupError

# Reproducibility
random.seed(42)
np.random.seed(42)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\abaiy\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\abaiy\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\abaiy\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\abaiy\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


# Basic Data Inspection and Pre processing

In [30]:
df = pd.read_csv("Reviews.csv", encoding='latin1')
df.head()

,Location_Name,Located_City,Location,Location_Type,User_ID,User_Location,User_Locale,User_Contributions,Travel_Date,Published_Date,Rating,Helpful_Votes,Title,Text
0,Arugam Bay,Arugam Bay,"Arugam Bay, Eastern Province",Beaches,User 1,"Dunsborough, Australia",en_US,8,2019-07,2019-07-31T07:53:21-04:00,5,1,Best nail spa in Arugam bay on the water!,I had a manicure here and it really was profes...
1,Arugam Bay,Arugam Bay,"Arugam Bay, Eastern Province",Beaches,User 2,"Bendigo, Australia",en_US,4,2019-06,2019-07-21T21:50:11-04:00,4,0,Best for surfing,"Overall, it is a wonderful experience. We visi..."
2,Arugam Bay,Arugam Bay,"Arugam Bay, Eastern Province",Beaches,User 3,"Melbourne, Australia",en_US,13,2019-07,2019-07-15T18:52:55-04:00,5,0,We Love Arugam Bay,"Great place to chill, swim, surf, eat, shop, h..."
3,Arugam Bay,Arugam Bay,"Arugam Bay, Eastern Province",Beaches,User 4,"Ericeira, Portugal",en_US,4,2019-06,2019-07-03T10:32:41-04:00,5,0,Sun and waves.,Good place for surf and a few stores to going ...
4,Arugam Bay,Arugam Bay,"Arugam Bay, Eastern Province",Beaches,User 5,"Pistoia, Italy",en_US,14,2019-07,2019-07-02T17:07:02-04:00,5,0,"Great swimming, surfing, great fish aznd frien...",This place is great for surfing but even if yo...


In [31]:
df.shape

(16156, 14)

In [32]:
df.columns

Index(['Location_Name', 'Located_City', 'Location', 'Location_Type', 'User_ID',
       'User_Location', 'User_Locale', 'User_Contributions', 'Travel_Date',
       'Published_Date', 'Rating', 'Helpful_Votes', 'Title', 'Text'],
      dtype='str')

In [33]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 16156 entries, 0 to 16155
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   Location_Name       16156 non-null  str  
 1   Located_City        16156 non-null  str  
 2   Location            16156 non-null  str  
 3   Location_Type       16156 non-null  str  
 4   User_ID             16156 non-null  str  
 5   User_Location       16156 non-null  str  
 6   User_Locale         16156 non-null  str  
 7   User_Contributions  16156 non-null  int64
 8   Travel_Date         16156 non-null  str  
 9   Published_Date      16156 non-null  str  
 10  Rating              16156 non-null  int64
 11  Helpful_Votes       16156 non-null  int64
 12  Title               16156 non-null  str  
 13  Text                16156 non-null  str  
dtypes: int64(3), str(11)
memory usage: 1.7 MB


In [34]:
df.describe()

,User_Contributions,Rating,Helpful_Votes
count,16156.000000,16156.000000,16156.000000
mean,191.624845,4.167492,0.709458
std,500.100421,1.006840,3.672513
min,1.000000,1.000000,0.000000
25%,18.000000,4.000000,0.000000
50%,54.000000,4.000000,0.000000
75%,155.000000,5.000000,1.000000
max,9010.000000,5.000000,233.000000


In [35]:
df.isnull().sum()

Location_Name         0
Located_City          0
Location              0
Location_Type         0
User_ID               0
User_Location         0
User_Locale           0
User_Contributions    0
Travel_Date           0
Published_Date        0
Rating                0
Helpful_Votes         0
Title                 0
Text                  0
dtype: int64

In [36]:
df.duplicated().sum()

np.int64(0)

In [37]:
df["Rating"].value_counts()

Rating
5    7649
4    5196
3    2166
2     658
1     487
Name: count, dtype: int64

In [38]:
df["Location_Type"].value_counts()

Location_Type
Religious Sites            3017
Beaches                    2110
Farms                      1884
Nature & Wildlife Areas    1557
Museums                    1525
Historic Sites             1519
Gardens                    1354
National Parks             1205
Waterfalls                  933
Bodies of Water             839
Zoological Gardens          213
Name: count, dtype: int64

In [39]:
df["Located_City"].value_counts().head(10)

Located_City
Nuwara Eliya    2221
Anuradhapura    1758
Kandy           1480
Colombo         1171
Sigiriya         763
Habarana         754
Hikkaduwa        515
Galle            511
Jaffna           475
Ella             471
Name: count, dtype: int64

In [40]:
date_columns = ["Travel_Date", "Published_Date"]

for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors="coerce", utc=True)

df.dtypes

Location_Name                         str
Located_City                          str
Location                              str
Location_Type                         str
User_ID                               str
User_Location                         str
User_Locale                           str
User_Contributions                  int64
Travel_Date           datetime64[us, UTC]
Published_Date        datetime64[us, UTC]
Rating                              int64
Helpful_Votes                       int64
Title                                 str
Text                                  str
dtype: object

In [41]:
df = df.drop(columns=["User_ID"])

In [42]:
df["review_length"] = df["Text"].apply(len)          # Creating new features
df["word_count"] = df["Text"].apply(lambda x: len(str(x).split()))
df.info()
df.describe()

<class 'pandas.DataFrame'>
RangeIndex: 16156 entries, 0 to 16155
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype              
---  ------              --------------  -----              
 0   Location_Name       16156 non-null  str                
 1   Located_City        16156 non-null  str                
 2   Location            16156 non-null  str                
 3   Location_Type       16156 non-null  str                
 4   User_Location       16156 non-null  str                
 5   User_Locale         16156 non-null  str                
 6   User_Contributions  16156 non-null  int64              
 7   Travel_Date         16156 non-null  datetime64[us, UTC]
 8   Published_Date      16156 non-null  datetime64[us, UTC]
 9   Rating              16156 non-null  int64              
 10  Helpful_Votes       16156 non-null  int64              
 11  Title               16156 non-null  str                
 12  Text                16156 non-null  str    

,User_Contributions,Rating,Helpful_Votes,review_length,word_count
count,16156.000000,16156.000000,16156.000000,16156.000000,16156.000000
mean,191.624845,4.167492,0.709458,380.618779,70.068705
std,500.100421,1.006840,3.672513,329.177103,60.678855
min,1.000000,1.000000,0.000000,50.000000,1.000000
25%,18.000000,4.000000,0.000000,176.000000,32.000000
50%,54.000000,4.000000,0.000000,282.000000,52.000000
75%,155.000000,5.000000,1.000000,466.000000,86.000000
max,9010.000000,5.000000,233.000000,9430.000000,1700.000000


In [43]:
for value in df["Location_Name"].unique():
    print(value)

Arugam Bay
Bentota Beach
Hikkaduwa Beach
Jungle Beach
Kalutara Beach
Marble Beach
Mirissa Beach
Mount Lavinia Beach
Negombo Beach
Nilaveli Beach
Passikudah Beach
Gregory Lake
Kandy Lake
Tissa Wewa
Twin Baths (Kuttam Pokuna)
Ambewela Farms
Bluefield Tea Gardens
Dambatenne Tea Factory
Damro Labookellie Tea Centre and Tea Garden
Glenloch Tea Factory
Halpewatte Tea Factory Tour
Handunugoda Tea Estate
Pedro Tea Factory
Brief Garden - Bevis Bawa
Hakgala Botanic Gardens
New Ranweli Spice Garden
Royal Botanical Gardens
Victoria Park of Nuwara Eliya
Galle Fort
Jaffna Fort
Liptons Seat
Polonnaruwa
Ritigala Forest Monastery
Sigiriya The Ancient Rock Fortress
Ariyapala Mask Museum
Ceylon Tea Museum
Colombo National Museum
Community Tsunami Museum
Martin Wickramasinghe Folk Museum Complex
Sigiriya Museum
World Buddhist Museum
Bundala National Park
Horton Plains National Park
Kaudulla National Park
Kumana National Park
Minneriya National Park
Pigeon Island National Park
Udawalawe National Park
Kala

In [44]:
for value in df["Located_City"].unique():
    print(value)

Arugam Bay
Bentota
Hikkaduwa
Unawatuna
Kalutara
Trincomalee
Mirissa
Colombo
Negombo
Nilaveli
Kalkudah
Nuwara Eliya
Kandy
Tissamaharama
Anuradhapura
Haputale
Katukitula
Ella
Ahangama
Beruwala
Peradeniya
Galle
Jaffna
Polonnaruwa
Sigiriya
Ambalangoda
Weligatta
Habarana
Ampara
Embilipitiya
Kalametiya
Pinnawala
Deniyaya
Saliyapura
Koslanda
Pussellawa


In [45]:
# Shared location-engineering helpers (define once before validation/extraction cells)
import re

provinces = [
    "Western Province",
    "Central Province",
    "Southern Province",
    "Northern Province",
    "Eastern Province",
    "North Western Province",
    "North Central Province",
    "Uva Province",
    "Sabaragamuwa Province",
]

city_to_district = {
    "Arugam Bay": "Ampara",
    "Colombo": "Colombo",
    "Kandy": "Kandy",
    "Nuwara Eliya": "Nuwara Eliya",
    "Galle": "Galle",
    "Mirissa": "Matara",
    "Ella": "Badulla",
    "Negombo": "Gampaha",
    "Polonnaruwa": "Polonnaruwa",
    "Sigiriya": "Matale",
    "Trincomalee": "Trincomalee",
    "Jaffna": "Jaffna",
    "Batticaloa": "Batticaloa",
    "Anuradhapura": "Anuradhapura",
    "Matara": "Matara",
    "Kalutara": "Kalutara",
    "Bentota": "Galle",
    "Hikkaduwa": "Galle",
    "Mirigama": "Gampaha",
    "Habarana": "Anuradhapura",
}

manual_location_mappings = {
    "Udawalawe National Park": {
        "province": "Sabaragamuwa Province",
        "district": "Ratnapura",
    },
    "North Central Province": {
        "province": "North Central Province",
        "district": "Anuradhapura",
    },
}

province_pattern = re.compile(
    r"(" + "|".join([re.escape(p) for p in provinces]) + r")", flags=re.IGNORECASE
)


def extract_province(location_str):
    if not isinstance(location_str, str) or not location_str.strip():
        return None

    loc = location_str.strip()

    for k, v in manual_location_mappings.items():
        if k.lower() == loc.lower():
            return v.get("province")

    m = province_pattern.search(loc)
    if m:
        matched = m.group(1)
        for p in provinces:
            if p.lower() == matched.lower():
                return p
        return matched

    m2 = re.search(r"([A-Za-z ]+Province)\\b", loc)
    if m2:
        return m2.group(1).strip()

    parts = [p.strip() for p in loc.split(",") if p.strip()]
    if parts:
        last = parts[-1]
        for p in provinces:
            if (
                last.lower() == p.replace(" Province", "").lower()
                or last.lower() == p.lower()
            ):
                return p

    return None


def infer_district(row):
    city = row.get("Located_City")
    location = row.get("Location")

    if isinstance(location, str):
        loc = location.strip()
        for k, v in manual_location_mappings.items():
            if k.lower() == loc.lower():
                return v.get("district")

    if isinstance(city, str) and city in city_to_district:
        return city_to_district[city]

    if not isinstance(location, str) or not location.strip():
        return None

    parts = [p.strip() for p in location.split(",") if p.strip()]

    for part in parts:
        if "District" in part:
            return part.replace("District", "").strip()

    if len(parts) == 1 and (
        any(parts[0].lower() == p.lower() for p in provinces)
        or "province" in parts[0].lower()
    ):
        if isinstance(city, str) and city.strip():
            if city in city_to_district:
                return city_to_district[city]
            return city
        return None

    if len(parts) >= 3:
        return parts[-2]

    if len(parts) == 2:
        first = parts[0]
        if city and isinstance(city, str) and city.lower() == first.lower():
            return city_to_district.get(city) or first
        return first

    if len(parts) == 1:
        single = parts[0]
        if (
            any(p.lower() == single.lower() for p in provinces)
            or "province" in single.lower()
        ):
            return None
        return single

    return None

In [46]:
# Validation-only cell: use the single canonical functions defined earlier (do not redefine them)
# This computes checks and lists remaining unmapped locations for review

# Compute check columns without redefining functions
df['province_check'] = df['Location'].apply(extract_province)
df['district_check'] = df.apply(infer_district, axis=1)

# Rows where either province or district could not be inferred
unmapped = df[df['province_check'].isnull() | df['district_check'].isnull()]

print("Unique Locations with missing province or district (examples):")
print(sorted(unmapped['Location'].dropna().unique())[:200])
print('\nCount of rows with missing province or district:', len(unmapped))

# show a few distinct examples for manual mapping decisions
display(unmapped[['Location','Located_City','province_check','district_check']].drop_duplicates().head(50))

Unique Locations with missing province or district (examples):
[]

Count of rows with missing province or district: 0


,Location,Located_City,province_check,district_check


## Location Feature Engineering (Province and District)

This cell adds `province` and `district` columns from `Location` and `Located_City`, then saves the enriched dataset to `processed_tourism_reviews_with_locations.csv`.

In [47]:
from pathlib import Path
import pandas as pd
import re

# Resolve paths for notebook execution context
project_dir = Path.cwd()
csv_in = project_dir / "Reviews.csv"
csv_out = project_dir / "processed_tourism_reviews_with_locations.csv"

df = pd.read_csv(csv_in, encoding="latin1")

# Known Sri Lankan provinces (standard names)
provinces = [
    "Western Province",
    "Central Province",
    "Southern Province",
    "Northern Province",
    "Eastern Province",
    "North Western Province",
    "North Central Province",
    "Uva Province",
    "Sabaragamuwa Province",
]

# Small city -> district mapping (best-effort for common locations seen in dataset)
city_to_district = {
    "Arugam Bay": "Ampara",
    "Colombo": "Colombo",
    "Kandy": "Kandy",
    "Nuwara Eliya": "Nuwara Eliya",
    "Galle": "Galle",
    "Mirissa": "Matara",
    "Ella": "Badulla",
    "Negombo": "Gampaha",
    "Polonnaruwa": "Polonnaruwa",
    "Sigiriya": "Matale",
    "Trincomalee": "Trincomalee",
    "Jaffna": "Jaffna",
    "Batticaloa": "Batticaloa",
    "Anuradhapura": "Anuradhapura",
    "Matara": "Matara",
    "Kalutara": "Kalutara",
    "Bentota": "Galle",
    "Hikkaduwa": "Galle",
    "Mirigama": "Gampaha",
    "Habarana": "Anuradhapura",
}

# Manual mappings for special Location strings
manual_location_mappings = {
    "Udawalawe National Park": {
        "province": "Sabaragamuwa Province",
        "district": "Ratnapura",
    },
    "North Central Province": {
        "province": "North Central Province",
        "district": "Anuradhapura",
    },
}

province_pattern = re.compile(
    r"(" + "|".join([re.escape(p) for p in provinces]) + r")", flags=re.IGNORECASE
)


def extract_province(location_str):
    if not isinstance(location_str, str) or not location_str.strip():
        return None

    loc = location_str.strip()

    # Manual mapping exact match (case-insensitive)
    for k, v in manual_location_mappings.items():
        if k.lower() == loc.lower():
            return v.get("province")

    # Direct match using known province names
    m = province_pattern.search(loc)
    if m:
        matched = m.group(1)
        for p in provinces:
            if p.lower() == matched.lower():
                return p
        return matched

    # Catch trailing "Province" text
    m2 = re.search(r"([A-Za-z ]+Province)\\b", loc)
    if m2:
        return m2.group(1).strip()

    # Comma-separated: last token might be province name without "Province"
    parts = [p.strip() for p in loc.split(",") if p.strip()]
    if parts:
        last = parts[-1]
        for p in provinces:
            if last.lower() == p.replace(" Province", "").lower() or last.lower() == p.lower():
                return p

    return None


def infer_district(row):
    city = row.get("Located_City")
    location = row.get("Location")

    # Manual mapping exact match
    if isinstance(location, str):
        loc = location.strip()
        for k, v in manual_location_mappings.items():
            if k.lower() == loc.lower():
                return v.get("district")

    # Prefer explicit city mapping
    if isinstance(city, str) and city in city_to_district:
        return city_to_district[city]

    if not isinstance(location, str) or not location.strip():
        return None

    parts = [p.strip() for p in location.split(",") if p.strip()]

    # If any part explicitly mentions "District", use it
    for part in parts:
        if "District" in part:
            return part.replace("District", "").strip()

    # If location is just a province name, try to use city as district
    if len(parts) == 1 and (
        any(parts[0].lower() == p.lower() for p in provinces)
        or "province" in parts[0].lower()
    ):
        if isinstance(city, str) and city.strip():
            if city in city_to_district:
                return city_to_district[city]
            return city
        return None

    # If parts are like "Town, District, Province" -> pick middle as district
    if len(parts) >= 3:
        return parts[-2]

    # If len==2 and first is likely a district/city, return first
    if len(parts) == 2:
        first = parts[0]
        if city and isinstance(city, str) and city.lower() == first.lower():
            return city_to_district.get(city) or first
        return first

    # Fallback: if the single token is not a province, return it
    if len(parts) == 1:
        single = parts[0]
        if any(p.lower() == single.lower() for p in provinces) or "province" in single.lower():
            return None
        return single

    return None


print("Extracting province and district (best-effort)...")

df["province"] = df["Location"].apply(extract_province)
df["district"] = df.apply(infer_district, axis=1)

# Normalize extracted text fields
for col in ["province", "district"]:
    df[col] = df[col].astype(object).apply(lambda x: x.strip() if isinstance(x, str) else x)

print("Province value counts (top 20):")
print(df["province"].value_counts(dropna=False).head(20))

print("\nSample districts (top 20):")
print(df["district"].value_counts(dropna=False).head(20))

print(f"Writing: {csv_out}")
df.to_csv(csv_out, index=False)
print("Done.")

Extracting province and district (best-effort)...
Province value counts (top 20):
province
Central Province          5252
North Central Province    3171
Southern Province         2648
Western Province          1890
Eastern Province          1162
Uva Province              1040
Sabaragamuwa Province      518
Northern Province          475
Name: count, dtype: int64

Sample districts (top 20):
district
Anuradhapura    2838
Kandy           2342
Nuwara Eliya    2221
Galle           1823
Colombo         1171
Matale           763
Jaffna           475
Badulla          471
Trincomalee      409
Haputale         409
Nilaveli         371
Pinnawala        369
Polonnaruwa      333
Beruwala         292
Ampara           222
Kalutara         218
Matara           217
Gampaha          209
Deniyaya         196
Weligatta        189
Name: count, dtype: int64
Writing: c:\Users\abaiy\Group-J---Research-Paper\processed_tourism_reviews_with_locations.csv
Done.


# User Location Parsing - Country & Region Extraction

This section extracts sovereign country and state/region information from user location strings using rule-based parsing with optional LLM fallback for unresolved locations.

In [48]:
# Install required packages for location parsing
import subprocess
import sys

required_packages = ["pycountry"]

for package in required_packages:
    try:
        __import__(package)
        print(f"✓ {package} already installed")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        print(f"✓ {package} installed successfully")

✓ pycountry already installed


In [49]:
# Import required libraries
import argparse
import importlib
import json
import os
import re
from dataclasses import dataclass
from typing import Dict, Optional, Tuple
from urllib import error, request

import pycountry

In [50]:
# Define ParseResult dataclass
@dataclass
class ParseResult:
    country: Optional[str]
    other: Optional[str]
    method: str
    confidence: float

In [51]:
# Define regional collections (states, provinces)
US_STATES = {
    "alabama", "alaska", "arizona", "arkansas", "california", "colorado",
    "connecticut", "delaware", "florida", "georgia", "hawaii", "idaho",
    "illinois", "indiana", "iowa", "kansas", "kentucky", "louisiana",
    "maine", "maryland", "massachusetts", "michigan", "minnesota",
    "mississippi", "missouri", "montana", "nebraska", "nevada",
    "new hampshire", "new jersey", "new mexico", "new york",
    "north carolina", "north dakota", "ohio", "oklahoma", "oregon",
    "pennsylvania", "rhode island", "south carolina", "south dakota",
    "tennessee", "texas", "utah", "vermont", "virginia", "washington",
    "west virginia", "wisconsin", "wyoming", "district of columbia", "puerto rico",
}

AU_STATES = {
    "new south wales", "queensland", "south australia", "tasmania",
    "victoria", "western australia", "australian capital territory",
    "northern territory",
}

CA_PROVINCES = {
    "alberta", "british columbia", "manitoba", "new brunswick",
    "newfoundland and labrador", "nova scotia", "ontario",
    "prince edward island", "quebec", "saskatchewan",
    "northwest territories", "nunavut", "yukon",
}

INDIA_STATES = {
    "andhra pradesh", "arunachal pradesh", "assam", "bihar",
    "chhattisgarh", "goa", "gujarat", "haryana", "himachal pradesh",
    "jharkhand", "karnataka", "kerala", "madhya pradesh",
    "maharashtra", "manipur", "meghalaya", "mizoram", "nagaland",
    "odisha", "punjab", "rajasthan", "sikkim", "tamil nadu",
    "telangana", "tripura", "uttar pradesh", "uttarakhand",
    "west bengal", "delhi", "jammu and kashmir", "ladakh",
}

UK_REGIONS = {"england", "scotland", "wales", "northern ireland"}

REGION_TO_COUNTRY = {
    **{state: "United States" for state in US_STATES},
    **{state: "Australia" for state in AU_STATES},
    **{state: "Canada" for state in CA_PROVINCES},
    **{state: "India" for state in INDIA_STATES},
    **{region: "United Kingdom" for region in UK_REGIONS},
    "new england": "United States",
}

In [52]:
# Utility and helper functions
def normalize_space(value: str) -> str:
    return re.sub(r"\s+", " ", value).strip()


def clean_token(token: str) -> str:
    cleaned = re.sub(r"[^\w\s\-']", " ", token, flags=re.UNICODE)
    return normalize_space(cleaned)


def title_case_token(token: str) -> str:
    return " ".join(part.capitalize() for part in token.split())


def build_country_index() -> Tuple[Dict[str, str], Dict[str, str]]:
    """Build country name index and preferred names mapping from pycountry."""
    country_alias_to_name: Dict[str, str] = {}

    for country in pycountry.countries:
        canonical = country.name
        country_alias_to_name[canonical.lower()] = canonical
        if hasattr(country, "official_name"):
            country_alias_to_name[country.official_name.lower()] = canonical
        if hasattr(country, "common_name"):
            country_alias_to_name[country.common_name.lower()] = canonical

    manual_aliases = {
        "usa": "United States", "u.s.a": "United States", "us": "United States",
        "u.s": "United States", "united states of america": "United States",
        "uk": "United Kingdom", "u.k": "United Kingdom",
        "england": "United Kingdom", "scotland": "United Kingdom",
        "wales": "United Kingdom", "northern ireland": "United Kingdom",
        "uae": "United Arab Emirates", "u.a.e": "United Arab Emirates",
        "the netherlands": "Netherlands", "holland": "Netherlands",
        "south korea": "Korea, Republic of",
        "north korea": "Korea, Democratic People's Republic of",
        "russia": "Russian Federation", "viet nam": "Vietnam",
    }
    country_alias_to_name.update(manual_aliases)

    country_short_to_preferred = {
        "Korea, Republic of": "South Korea",
        "Korea, Democratic People's Republic of": "North Korea",
        "Russian Federation": "Russia", "Viet Nam": "Vietnam",
        "Iran, Islamic Republic of": "Iran",
        "Syrian Arab Republic": "Syria",
        "Moldova, Republic of": "Moldova",
        "Tanzania, United Republic of": "Tanzania",
        "Venezuela, Bolivarian Republic of": "Venezuela",
        "Bolivia, Plurinational State of": "Bolivia",
        "Brunei Darussalam": "Brunei",
        "Lao People's Democratic Republic": "Laos",
        "Czechia": "Czech Republic",
        "United States": "United States",
        "United Kingdom": "United Kingdom",
    }

    return country_alias_to_name, country_short_to_preferred


COUNTRY_ALIAS_TO_NAME, COUNTRY_SHORT_TO_PREFERRED = build_country_index()


def to_preferred_country_name(canonical_name: str) -> str:
    return COUNTRY_SHORT_TO_PREFERRED.get(canonical_name, canonical_name)

In [53]:
# Main deterministic location parsing function
def parse_user_location_deterministic(raw_value: object) -> ParseResult:
    """Parse user location using rule-based pattern matching."""
    if raw_value is None or (isinstance(raw_value, float) and pd.isna(raw_value)):
        return ParseResult(country=None, other=None, method="empty", confidence=0.0)

    raw_text = str(raw_value).strip()
    if not raw_text:
        return ParseResult(country=None, other=None, method="empty", confidence=0.0)

    parts = [clean_token(part) for part in raw_text.split(",")]
    parts = [part for part in parts if part]
    if not parts:
        return ParseResult(country=None, other=None, method="empty", confidence=0.0)

    country = None
    country_idx = None

    # Search for country name from right to left
    for idx in range(len(parts) - 1, -1, -1):
        token_lower = parts[idx].lower()
        if token_lower in COUNTRY_ALIAS_TO_NAME:
            canonical = COUNTRY_ALIAS_TO_NAME[token_lower]
            country = to_preferred_country_name(canonical)
            country_idx = idx
            break

    # Search for region/state that maps to a country
    region_token = None
    region_country = None
    for idx in range(len(parts) - 1, -1, -1):
        token_lower = parts[idx].lower()
        if token_lower in REGION_TO_COUNTRY:
            region_token = title_case_token(token_lower)
            region_country = REGION_TO_COUNTRY[token_lower]
            if region_country == "United Kingdom" and token_lower in UK_REGIONS:
                region_token = token_lower.title()
            break

    # Handle region-inferred country
    if not country and region_country:
        country = region_country
        if region_token:
            return ParseResult(
                country=country, other=region_token,
                method="rule_region_infer_country", confidence=0.88,
            )
        return ParseResult(
            country=country, other=None,
            method="rule_region_infer_country", confidence=0.8,
        )

    # Handle explicit country with optional region
    if country:
        if region_token and REGION_TO_COUNTRY.get(region_token.lower()) == country:
            return ParseResult(
                country=country, other=region_token,
                method="rule_country_and_region", confidence=0.98,
            )

        if country_idx is not None:
            for idx in range(country_idx - 1, -1, -1):
                token_lower = parts[idx].lower()
                if token_lower in REGION_TO_COUNTRY and REGION_TO_COUNTRY[token_lower] == country:
                    return ParseResult(
                        country=country, other=title_case_token(token_lower),
                        method="rule_country_plus_left_region", confidence=0.95,
                    )

        return ParseResult(
            country=country, other=None,
            method="rule_country_only", confidence=0.9,
        )

    return ParseResult(country=None, other=None, method="unresolved", confidence=0.0)

In [54]:
# LLM fallback function for unresolved locations
def call_llm_for_location(
    raw_location: str,
    api_key: str,
    api_base_url: str,
    model: str,
    timeout_seconds: int,
) -> Optional[ParseResult]:
    """Use LLM API to parse unresolved location strings."""
    prompt = (
        "Extract a sovereign country and a state/region value from this user location text. "
        "Rules: (1) other should be state/region only, never city. "
        "(2) map subregions to sovereign country, e.g., England->United Kingdom. "
        "(3) if unknown, use null. "
        "Return strict JSON only with keys: country, other, confidence.\n"
        f"location: {raw_location}"
    )

    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": "You are a precise location normalizer."},
            {"role": "user", "content": prompt},
        ],
        "temperature": 0,
        "response_format": {"type": "json_object"},
    }

    body = json.dumps(payload).encode("utf-8")
    req = request.Request(
        f"{api_base_url.rstrip('/')}/chat/completions",
        data=body,
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {api_key}",
        },
        method="POST",
    )

    try:
        with request.urlopen(req, timeout=timeout_seconds) as resp:
            response_json = json.loads(resp.read().decode("utf-8"))
        content = response_json["choices"][0]["message"]["content"]
        parsed = json.loads(content)
        country = parsed.get("country")
        other = parsed.get("other")
        confidence = float(parsed.get("confidence", 0.6))

        if country is not None and str(country).strip() == "":
            country = None
        if other is not None and str(other).strip() == "":
            other = None

        if country:
            normalized_country = COUNTRY_ALIAS_TO_NAME.get(str(country).lower(), country)
            country = to_preferred_country_name(normalized_country)

        if other:
            other_token = clean_token(str(other))
            if not other_token:
                other = None
            else:
                other = title_case_token(other_token.lower())

        return ParseResult(
            country=country, other=other, method="llm_fallback",
            confidence=max(0.0, min(confidence, 1.0)),
        )
    except (error.URLError, error.HTTPError, ValueError, KeyError, TypeError):
        return None

In [55]:
# Main processing function
def run(
    input_csv: str,
    output_csv: str,
    location_column: str,
    use_llm: bool,
    llm_api_key: str,
    llm_api_base_url: str,
    llm_model: str,
    llm_timeout: int,
) -> None:
    """Process CSV file and extract country/region from location column."""
    df = pd.read_csv(input_csv, encoding="latin1")

    if location_column not in df.columns:
        raise ValueError(f"Column '{location_column}' not found in input CSV.")

    # Apply deterministic parsing
    parsed = df[location_column].apply(parse_user_location_deterministic)
    df["user_country"] = parsed.apply(lambda x: x.country)
    df["user_other"] = parsed.apply(lambda x: x.other)
    df["user_location_parse_method"] = parsed.apply(lambda x: x.method)
    df["user_location_parse_confidence"] = parsed.apply(lambda x: x.confidence)

    llm_rows = 0
    api_key = llm_api_key.strip()

    # Apply LLM fallback for unresolved rows
    if use_llm:
        if not api_key:
            print("LLM API key not found. Skipping LLM fallback.")
        else:
            unresolved_mask = df["user_country"].isna()
            unresolved_values = (
                df.loc[unresolved_mask, location_column].dropna().astype(str).str.strip()
            )
            unresolved_values = unresolved_values[unresolved_values != ""]

            unique_unresolved = unresolved_values.drop_duplicates().tolist()
            cache: Dict[str, ParseResult] = {}

            for raw_location in unique_unresolved:
                result = call_llm_for_location(
                    raw_location=raw_location, api_key=api_key,
                    api_base_url=llm_api_base_url, model=llm_model,
                    timeout_seconds=llm_timeout,
                )
                if result and result.country:
                    cache[raw_location] = result

            def apply_llm(row):
                nonlocal llm_rows
                current_country = row["user_country"]
                location_value = row[location_column]
                if pd.notna(current_country):
                    return row
                if pd.isna(location_value):
                    return row
                key = str(location_value).strip()
                result = cache.get(key)
                if not result:
                    return row
                row["user_country"] = result.country
                row["user_other"] = result.other
                row["user_location_parse_method"] = result.method
                row["user_location_parse_confidence"] = result.confidence
                llm_rows += 1
                return row

            df = df.apply(apply_llm, axis=1)

    # Save output
    output_dir = os.path.dirname(os.path.abspath(output_csv))
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)

    df.to_csv(output_csv, index=False)

    # Print statistics
    total_rows = len(df)
    resolved_rows = int(df["user_country"].notna().sum())
    unresolved_rows = total_rows - resolved_rows
    method_counts = df["user_location_parse_method"].value_counts(dropna=False)

    print(f"Input rows: {total_rows}")
    print(f"Resolved country rows: {resolved_rows}")
    print(f"Unresolved rows: {unresolved_rows}")
    print(f"Resolved ratio: {resolved_rows / total_rows:.2%}" if total_rows else "Resolved ratio: 0.00%")
    print(f"LLM-filled rows: {llm_rows}")
    print("Method counts:")
    print(method_counts)
    print(f"Wrote output: {output_csv}")

## Example Usage

Extract country and region from the `User_Location` column in the dataset.

In [56]:
# Example: Process the tourism reviews and extract country/region
input_file = "processed_tourism_reviews_with_locations.csv"
output_file = "processed_tourism_reviews_country_other.csv"

# Load the most recent processed dataset
if os.path.exists(input_file):
    # Process with deterministic parsing only (set use_llm=True to enable LLM fallback)
    run(
        input_csv=input_file,
        output_csv=output_file,
        location_column="User_Location",
        use_llm=False,  # Set to True and provide API key for LLM fallback
        llm_api_key="",  # Set your API key here if using LLM
        llm_api_base_url="https://openrouter.ai/api/v1",
        llm_model="google/gemma-2-9b-it:free",
        llm_timeout=20,
    )
    print(f"✓ Location parsing complete. Results saved to: {output_file}")
else:
    print(f"Error: Input file '{input_file}' not found.")

Input rows: 16156
Resolved country rows: 15114
Unresolved rows: 1042
Resolved ratio: 93.55%
LLM-filled rows: 0
Method counts:
user_location_parse_method
rule_country_only            13954
unresolved                    1041
rule_region_infer_country      921
rule_country_and_region        239
empty                            1
Name: count, dtype: int64
Wrote output: processed_tourism_reviews_country_other.csv
✓ Location parsing complete. Results saved to: processed_tourism_reviews_country_other.csv
